# Baseline resource calculations

The results of this notebook will become the Kubernetes resource requests.

For this calculation, the 15th percentile of user load was calculated for each of four roles: back-office, portfolio manager, trader, and investor.  The it-operations role has one fixed user.  The calculation of the 15th percentile load is in the [user_load_analysis.ipynb](./user_load_analysis.ipynb) notebook.  This spreadsheet produces a json file containing descriptive satistics on CPU utilization.  That spreadsheet is used as input to analysis.ipynb in the globeco-helm repo to calculate actual resource requests for each microservice.

In [1]:
%load_ext autoreload
%autoreload 2

In [50]:
import json
import os
import datetime

from dotenv import load_dotenv
import pandas as pd

import common

## Environmental Variables

In [3]:
load_dotenv()
s3_bucket = os.environ.get("S3_BUCKET")
print(f"S3 Bucket: {s3_bucket}")

S3 Bucket: kasbench-test-20260528-377288663341-us-east-1-an


## Constants

In [10]:
RUN = "calibration-15-percentile-a"
DATA_DIR = f"../data/{RUN}"
os.makedirs(DATA_DIR, exist_ok=True)
RUN_DB_PATH = f"{DATA_DIR}/logs.db"
print(f"Run database is {RUN_DB_PATH}")

Run database is ../data/calibration-15-percentile-a/logs.db


## Preprocess Locust DBs into a dataframe

In [18]:
# Download the locust database files
local_db_paths = common.download_locust_dbs(s3_bucket, RUN, DATA_DIR)
# Create the run database from the local files
common.create_run_db(RUN, RUN_DB_PATH, s3_bucket, local_db_paths)
# Convert the database into a dataframe
run_db_df = common.get_log_as_dataframe(RUN_DB_PATH)


We will create two summarized dataframes.  The dataframe to be used to calculate response time will exclude requests with a status code of 0.  This includes refused connections and network blips.  Refused connections may be caused by threadpool exhaustion or application failure, which are likely attributable to shortcomings of the autoscaler under test, while the network blips are random events equally likely to strike any autoscaler.  Requests with status code 0 have response times that are not indicative of the true responsiveness of the application.  They reflect the length of timeouts, which, in the case of network blips are opaque.  Excluding these transactions allows response time to better reflect actual application performance resulting from autoscaler actions.  Failure rate will include these requests, including both connection refused and intermittent network blips.  Although these intermittent network blips are not caused by autoscaler behavior, they are random and expected to average out.

In [28]:
run_db_df

,run_id,trial_id,role,autoscaler,user_id,request_type,name,response_time,response_length,response,status_code,reason,exception,start_time,url,method_and_name,success,failure
0,calibration-15-percentile-a,trial0001,back-office,none,4006e343-8b42-4348-a417-67f40574c898,POST,/api/portfolios/bulk,38.357512,139,<Response [201]>,201,Created,None,2026-09-16 15:27:31.028108,http://kasb-20260916151225958500000012-f616252...,POST /api/portfolios/bulk,1,0
1,calibration-15-percentile-a,trial0001,back-office,none,57c3fb63-6aa2-492b-b7b2-c41dcab2b5bb,POST,/api/portfolios/bulk,16.040725,140,<Response [201]>,201,Created,None,2026-09-16 15:27:31.224712,http://kasb-20260916151225958500000012-f616252...,POST /api/portfolios/bulk,1,0
2,calibration-15-percentile-a,trial0001,back-office,none,57c3fb63-6aa2-492b-b7b2-c41dcab2b5bb,GET,/api/portfolios/{id},10.809626,133,<Response [200]>,200,OK,None,2026-09-16 15:27:31.688127,http://kasb-20260916151225958500000012-f616252...,GET /api/portfolios/{id},1,0
3,calibration-15-percentile-a,trial0001,back-office,none,4006e343-8b42-4348-a417-67f40574c898,GET,/api/portfolios/{id},8.676832,132,<Response [200]>,200,OK,None,2026-09-16 15:27:31.850617,http://kasb-20260916151225958500000012-f616252...,GET /api/portfolios/{id},1,0
4,calibration-15-percentile-a,trial0001,back-office,none,4006e343-8b42-4348-a417-67f40574c898,POST,/api/transactions,33.522982,344,<Response [201]>,201,Created,None,2026-09-16 15:27:32.550614,http://kasb-20260916151225958500000012-f616252...,POST /api/transactions,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
618288,calibration-15-percentile-a,trial0007,trader,none,179c8d51-1d4f-475a-8aa4-a6ddafd7eca3,POST,/api/trade-orders/batch/submit,27.049779,1053,<Response [200]>,200,OK,None,2026-09-16 21:00:23.018494,http://kasb-20260916201525378300000012-ba5ea6d...,POST /api/trade-orders/batch/submit,1,0
618289,calibration-15-percentile-a,trial0007,trader,none,179c8d51-1d4f-475a-8aa4-a6ddafd7eca3,GET,/api/executions/{id},13.255518,404,<Response [200]>,200,OK,None,2026-09-16 21:00:23.050712,http://kasb-20260916201525378300000012-ba5ea6d...,GET /api/executions/{id},1,0
618290,calibration-15-percentile-a,trial0007,trader,none,dfe67c71-2f1b-400d-957e-bbc7aa5db8ea,GET,/api/trades?orderId,6.632634,628,<Response [200]>,200,OK,None,2026-09-16 21:00:23.115371,http://kasb-20260916201525378300000012-ba5ea6d...,GET /api/trades?orderId,1,0
618291,calibration-15-percentile-a,trial0007,trader,none,dfe67c71-2f1b-400d-957e-bbc7aa5db8ea,POST,/api/trade-orders/batch/submit,17.509153,1054,<Response [200]>,200,OK,None,2026-09-16 21:00:23.127449,http://kasb-20260916201525378300000012-ba5ea6d...,POST /api/trade-orders/batch/submit,1,0


In [29]:
failure_rate_df = run_db_df
response_time_df = run_db_df[run_db_df["status_code"] != "0"]

## Summarize the Locust results by trial and rolled up

### Response time summary

In [30]:
trial_summary_rt_df = common.get_trial_summary_df(response_time_df)
trial_summary_rt_df

,autoscaler,trial_id,requests,mean_response_time,failures,failure_rate
0,none,trial0001,92593,21.513182,2,0.000022
1,none,trial0002,94352,24.370080,3,0.000032
2,none,trial0003,93233,27.422055,3,0.000032
3,none,trial0004,93815,31.182108,4,0.000043
4,none,trial0005,57043,25.446045,2,0.000035
5,none,trial0006,94360,26.097372,2,0.000021
6,none,trial0007,92810,26.412773,1,0.000011


In [31]:
trial_summary_rt_df.mean_response_time.describe()

count     7.000000
mean     26.063374
std       2.947408
min      21.513182
25%      24.908063
50%      26.097372
75%      26.917414
max      31.182108
Name: mean_response_time, dtype: float64

### Failure rate summary

In [32]:
trial_summary_fr_df = common.get_trial_summary_df(failure_rate_df)
trial_summary_fr_df

,autoscaler,trial_id,requests,mean_response_time,failures,failure_rate
0,none,trial0001,92593,21.513182,2,0.000022
1,none,trial0002,94352,24.370080,3,0.000032
2,none,trial0003,93233,27.422055,3,0.000032
3,none,trial0004,93815,31.182108,4,0.000043
4,none,trial0005,57130,155.098977,89,0.001558
5,none,trial0006,94360,26.097372,2,0.000021
6,none,trial0007,92810,26.412773,1,0.000011


In [33]:
trial_summary_fr_df.failure_rate.describe()

count    7.000000
mean     0.000245
std      0.000579
min      0.000011
25%      0.000021
50%      0.000032
75%      0.000037
max      0.001558
Name: failure_rate, dtype: float64

### Analysis

At the 15th percentile of user load for the baseline configuration, both response time and failure rate are well below the SLO of 100ms response time and 0.01% failure rate.  Resource limits based on actual resource utilization at this user-load is likely to require scaling for the upper 85% of baseline user-load.

## CPU metrics analysis for sizing

### CPU Usage

In [37]:
metric = "container_cpu_usage_seconds_total-container-_pod"
column_label = "cpu_cores"
cpu_usage_original_df = common.get_merged_range_metric_df(files=None, run_id=RUN, 
    s3_bucket=s3_bucket, metric=metric, column_label=column_label)
display(cpu_usage_original_df)

Getting run details for run_id: calibration-15-percentile-a, trial: trial0001
Getting run details for run_id: calibration-15-percentile-a, trial: trial0002
Getting run details for run_id: calibration-15-percentile-a, trial: trial0003
Getting run details for run_id: calibration-15-percentile-a, trial: trial0004
Getting run details for run_id: calibration-15-percentile-a, trial: trial0005
Getting run details for run_id: calibration-15-percentile-a, trial: trial0006
Getting run details for run_id: calibration-15-percentile-a, trial: trial0007


,timestamp,container,pod,cpu_cores,trial_id,autoscaler
0,2026-09-16 15:27:21.625000000,envoy,envoy-globeco-globeco-gateway-6bee274d-67566d6...,0.004513,trial0001,none
1,2026-09-16 15:27:36.625000000,envoy,envoy-globeco-globeco-gateway-6bee274d-67566d6...,0.003210,trial0001,none
2,2026-09-16 15:27:51.625000000,envoy,envoy-globeco-globeco-gateway-6bee274d-67566d6...,0.005435,trial0001,none
3,2026-09-16 15:28:21.625000000,envoy,envoy-globeco-globeco-gateway-6bee274d-67566d6...,0.020909,trial0001,none
4,2026-09-16 15:28:36.625000000,envoy,envoy-globeco-globeco-gateway-6bee274d-67566d6...,0.023240,trial0001,none
...,...,...,...,...,...,...
50370,2026-09-16 20:59:18.740999937,NaN,otel-collector-daemonset-collector-vgtdh,0.022516,trial0007,none
50371,2026-09-16 20:59:33.740999937,NaN,otel-collector-daemonset-collector-vgtdh,0.020628,trial0007,none
50372,2026-09-16 20:59:48.740999937,NaN,otel-collector-daemonset-collector-vgtdh,0.019211,trial0007,none
50373,2026-09-16 21:00:03.740999937,NaN,otel-collector-daemonset-collector-vgtdh,0.000000,trial0007,none


In [39]:
summary = cpu_usage_original_df.groupby('container').agg({
    'cpu_cores': ['mean', 'std', 'min', 'max'],
})
summary.columns = summary.columns.droplevel(0)
summary.columns.name = None
display(summary)
summary = summary.T

cpu_usage_json = summary.to_json()
with open('../../globeco-helm/data/cpu_usage.json', 'w') as f:
    f.write(cpu_usage_json)

,mean,std,min,max
container,,,,
envoy,0.025868,0.006762,0.002707,0.037873
envoy-gateway,0.001456,0.000383,0.000548,0.003519
globeco-allocation-service,0.000910,0.000301,0.000102,0.001823
globeco-allocation-service-postgresql,0.004713,0.000732,0.001663,0.006429
globeco-confirmation-service,0.011697,0.003085,0.003044,0.018257
globeco-debug-tools,0.000000,0.000000,0.000000,0.000000
globeco-execution-service,0.080357,0.030822,0.001940,0.344513
globeco-execution-service-postgresql,0.014507,0.003630,0.001369,0.024071
globeco-fix-engine,0.017485,0.007165,0.000075,0.077028


### Memory Usage

In [43]:
metric = "container_memory_max_usage_bytes-container"
column_label = "bytes"
memory_usage_original_df = common.get_merged_range_metric_df(files=None, run_id=RUN, 
    s3_bucket=s3_bucket, metric=metric, column_label=column_label)
memory_usage_original_df.bytes = memory_usage_original_df.bytes / 1024 / 1024
display(memory_usage_original_df)

,timestamp,bytes,container,trial_id,autoscaler
0,2026-09-16 15:27:21.625000000,4354.140625,NaN,trial0001,none
1,2026-09-16 15:27:36.625000000,4369.535156,NaN,trial0001,none
2,2026-09-16 15:27:51.625000000,4406.113281,NaN,trial0001,none
3,2026-09-16 15:28:06.625000000,4445.179688,NaN,trial0001,none
4,2026-09-16 15:28:21.625000000,4642.042969,NaN,trial0001,none
...,...,...,...,...,...
27388,2026-09-16 20:33:03.740999937,29.582031,wait-for-postgres,trial0007,none
27389,2026-09-16 20:33:18.740999937,23.511719,wait-for-postgres,trial0007,none
27390,2026-09-16 20:33:33.740999937,17.445312,wait-for-postgres,trial0007,none
27391,2026-09-16 20:33:48.740999937,6.152344,wait-for-postgres,trial0007,none


In [54]:
summary_memory = memory_usage_original_df.groupby('container').agg({
    'bytes': ['mean', 'std', 'min', 'max'],
})
summary_memory.columns = summary_memory.columns.droplevel(0)
summary_memory.columns.name = None
display(summary_memory)
summary_memory = summary_memory.T

memory_usage_json = summary_memory.to_json()
with open('../../globeco-helm/data/memory_usage.json', 'w') as f:
    f.write(memory_usage_json)

summary_memory = summary_memory.T

,mean,std,min,max
container,,,,
envoy,30.717782,1.571353,20.921875,31.953125
envoy-gateway,51.534640,1.141018,46.605469,53.523438
globeco-allocation-service,15.804987,2.157041,9.800781,18.308594
globeco-allocation-service-postgresql,107.491938,0.818462,104.136719,109.597656
globeco-confirmation-service,26.617441,5.582652,3.726562,34.929688
globeco-debug-tools,4.177455,0.341909,3.734375,4.691406
globeco-execution-service,272.074090,11.417840,208.039062,281.167969
globeco-execution-service-postgresql,141.045800,14.478502,96.980469,162.011719
globeco-fix-engine,17.967030,3.134754,7.214844,27.019531


In [48]:
memory = {'globeco-allocation-service': 100,
 'globeco-confirmation-service': 100,
 'globeco-execution-service': 700,
 'globeco-fix-engine': 100,
 'globeco-order-generation-service': 700,
 'globeco-order-service': 700,
 'globeco-portfolio-accounting-service': 100,
 'globeco-portfolio-management-portal': 200,
 'globeco-portfolio-service': 300,
 'globeco-pricing-service': 1000,
 'globeco-security-service': 200,
 'globeco-trade-service': 700}

microservices = ["globeco-allocation-service",
"globeco-confirmation-service",
"globeco-execution-service",
"globeco-fix-engine",
"globeco-order-generation-service",
"globeco-order-service",
"globeco-portfolio-accounting-service",
"globeco-portfolio-management-portal",
"globeco-portfolio-service",
"globeco-pricing-service",
"globeco-security-service",
"globeco-trade-service",]
len(microservices)

12

In [65]:
memory_series = pd.Series(memory, name="current_allocation")

# inner join summary_memory to memory_series
merged = summary_memory.merge(memory_series, left_index=True, right_index=True, how='inner')
merged["proposed"] = merged["max"] / 0.70
# round proposed up to the nearest 100
merged["proposed"] = merged["proposed"].apply(lambda x: int(x) if int(x) == x else int(x) + 1)
merged["proposed"] = merged["proposed"].apply(lambda x: x if x % 100 == 0 else x + (100 - x % 100))

merged

,mean,std,min,max,current_allocation,proposed
container,,,,,,
globeco-allocation-service,15.804987,2.157041,9.800781,18.308594,100,100
globeco-confirmation-service,26.617441,5.582652,3.726562,34.929688,100,100
globeco-execution-service,272.074090,11.417840,208.039062,281.167969,700,500
globeco-fix-engine,17.967030,3.134754,7.214844,27.019531,100,100
globeco-order-generation-service,223.609583,15.466229,178.265625,236.847656,700,400
globeco-order-service,603.273885,40.419627,458.554688,683.906250,700,1000
globeco-portfolio-accounting-service,32.663795,3.327098,3.722656,35.125000,100,100
globeco-portfolio-management-portal,266.714641,73.253404,138.425781,356.367188,200,600
globeco-portfolio-service,206.985007,63.809926,150.796875,367.300781,300,600
